# bigcompute.science Research Agent

Run the autonomous research agent from Google Colab. Monitors experiments, harvests results, runs multi-model peer reviews, fixes issues.

**You're in Colab = you have Google.** The easiest path is a free Gemini API key:

1. Go to [AI Studio](https://aistudio.google.com/apikey) (you're already logged into Google)
2. Click **Create API Key** (free, instant, no credit card)
3. Paste it below or add as a Colab Secret named `GEMINI_API_KEY`

Or use OpenAI/Anthropic keys if you have them. **Any one key works.**

> Part of [bigcompute.science](https://bigcompute.science) — open computational mathematics.

In [ ]:
# === Step 1: Clone + install ===
!git clone https://github.com/cahlen/idontknow.git 2>/dev/null || (cd idontknow && git pull)
%cd idontknow
!pip install -q httpx

In [ ]:
# === Step 2: Set your API key ===
# The agent tries providers in order: Anthropic -> OpenAI -> Gemini
# In Colab, Gemini is the easiest (free, you're already logged into Google)

import os

# --- Option A: Colab Secrets (recommended) ---
# Click the key icon in the sidebar, add GEMINI_API_KEY
try:
    from google.colab import userdata
    for key_name in ['GEMINI_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
        val = userdata.get(key_name, '')
        if val:
            os.environ[key_name] = val
            print(f'  Loaded {key_name} from Colab Secrets')
except:
    pass

# --- Option B: Paste directly ---
# Get a free Gemini key: https://aistudio.google.com/apikey
# os.environ['GEMINI_API_KEY'] = 'AIza...'       # Gemini (free)
# os.environ['OPENAI_API_KEY'] = 'sk-proj-...'   # OpenAI
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...' # Anthropic

has = {k: bool(os.environ.get(k)) for k in ['GEMINI_API_KEY','GOOGLE_API_KEY','OPENAI_API_KEY','ANTHROPIC_API_KEY']}
available = [k.replace('_API_KEY','') for k,v in has.items() if v]
print(f'\nAvailable: {", ".join(available) if available else "NONE"}')
if not available:
    print('\nGet a free Gemini key: https://aistudio.google.com/apikey')
    print('Then paste above or add as Colab Secret')

In [ ]:
# === Step 3: Quick test — can the agent talk to an LLM? ===
import sys
sys.path.insert(0, '.')
from scripts.research_agent import call_llm

result = call_llm('Respond with exactly: {"status": "ok", "model": "your-name"}', 'test')
if result:
    print(f'LLM connection: OK')
    print(f'Response: {result[:100]}')
else:
    print('FAILED — check your API key above')

In [ ]:
# === Step 4: Dry run — see what the agent would do ===
!python3 scripts/research_agent.py --once --dry-run

In [ ]:
# === Step 5: Harvest results (see what experiments completed) ===
!python3 scripts/research_agent.py --once --phase harvest

In [ ]:
# === Step 6: Peer review a finding ===
# Uses Gemini (free!) or whatever API key you have

finding = 'zaremba-density-phase-transition'  # @param {type:"string"}

# Determine which provider to use
if os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'):
    key = os.environ.get('GEMINI_API_KEY', os.environ.get('GOOGLE_API_KEY', ''))
    os.environ['API_KEY'] = key
    model, provider, api_base = 'gemini-2.5-flash', 'google', 'https://generativelanguage.googleapis.com/v1beta/openai'
elif os.environ.get('OPENAI_API_KEY'):
    os.environ['API_KEY'] = os.environ['OPENAI_API_KEY']
    model, provider, api_base = 'gpt-4.1', 'openai', 'https://api.openai.com/v1'
elif os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['API_KEY'] = os.environ['ANTHROPIC_API_KEY']
    model, provider, api_base = 'claude-sonnet-4-20250514', 'anthropic', 'https://api.anthropic.com/v1'
else:
    raise ValueError('No API key set')

print(f'Reviewing {finding} with {model} ({provider})...')
!python3 scripts/reviews/run_review.py --slug {finding} --model {model} --provider {provider} --api-base {api_base} --skip-existing

In [ ]:
# === Step 7: View all reviews for a finding ===
import json, glob

finding = 'zaremba-density-phase-transition'  # @param {type:"string"}

reviews = sorted(glob.glob(f'docs/verifications/{finding}*review*.json'))
print(f'{len(reviews)} review(s):\n')
for f in reviews:
    with open(f) as fh:
        r = json.load(fh)
    model = r.get('reviewer', {}).get('model', '?')
    verdict = r.get('overall_verdict', '?')
    cert = r.get('certification_recommendation', '?')
    print(f'  {model:25s} {verdict:25s} {cert}')

In [ ]:
# === Step 8: Rebuild manifest (aggregate all reviews) ===
!python3 scripts/reviews/aggregate.py
!python3 scripts/reviews/validate.py --all

In [ ]:
# === Step 9: Review ALL findings at once ===
# This will take a few minutes (15 findings x ~30s each)
# Gemini free tier: 15 RPM, so this stays within limits

if os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'):
    key = os.environ.get('GEMINI_API_KEY', os.environ.get('GOOGLE_API_KEY', ''))
    os.environ['API_KEY'] = key
    !python3 scripts/reviews/run_review.py --all --model gemini-2.5-flash --provider google --api-base https://generativelanguage.googleapis.com/v1beta/openai --skip-existing
elif os.environ.get('OPENAI_API_KEY'):
    os.environ['API_KEY'] = os.environ['OPENAI_API_KEY']
    !python3 scripts/reviews/run_review.py --all --model gpt-4.1 --provider openai --skip-existing
else:
    print('Set GEMINI_API_KEY or OPENAI_API_KEY first')

## Contribute Your Reviews

**Your review adds a new perspective.** Different models catch different errors. Claude found 6 issues in Round 1. o3-pro found the prime count error. Your Gemini review might find something they all missed.

**To submit:**
1. Download the review JSON(s) from `docs/verifications/`
2. Open a PR to [cahlen/idontknow](https://github.com/cahlen/idontknow)
3. Your review will be added to the audit ledger on [bigcompute.science/verification](https://bigcompute.science/verification/)

**To run the full autonomous loop** (needs a machine with GPUs):
```bash
export GEMINI_API_KEY='AIza...'  # or OPENAI_API_KEY or ANTHROPIC_API_KEY
nohup ./scripts/run_agent.sh --loop 10m &
```

**More info:** [AGENTS.md](https://github.com/cahlen/idontknow/blob/main/AGENTS.md) · [Verification](https://bigcompute.science/verification/) · [MCP Server](https://mcp.bigcompute.science/mcp) (23 tools)